# Evaluación Integral de Multiamenaza y Exposición de Edificaciones
### Iniciativa: *La Carpio, San José, Costa Rica* (Cluster Urbano Copernicus Centroamérica)

**Versión:** 1.1 — Auditada y alineada con los datos reales disponibles en el repositorio GitHub (2026-11-11)

**Equipo interinstitucional:**
| Nombre | Institución |
|---|---|
| Jonnathan Obando Artiaga | Ministerio de Vivienda y Asentamientos Humanos (MIVAH) |
| Victoria Delgado Fernández | Ministerio de Vivienda y Asentamientos Humanos (MIVAH) |
| Emiler Segura López | Instituto Nacional de Vivienda y Urbanismo (INVU) |
| Rodolfo Bolaños Bermúdez | Instituto Nacional de Vivienda y Urbanismo (INVU) |
| Jaqueline Vargas Bogantes | Municipalidad de San José |

---

Este cuaderno implementa el motor de **análisis de multiamenaza ponderada y exposición de edificaciones**,
diseñado para **omitir por completo los assets cerrados de Google Earth Engine (GEE)** y basarse en
las **capas abiertas, validadas y versionadas del repositorio público GitHub** del proyecto (`geovisor/capas/`).

### Principios del Modelo Abierto en GitHub:
1. **Independencia de GEE**: No requiere cuentas privadas de GEE. Todo el cálculo se realiza en Python abierto.
2. **Capas basadas en GitHub (`geovisor/capas/`)**: Lee directamente los GeoJSON/GeoPackages versionados por el equipo.
3. **Catálogo modular (`CATALOGO_AMENAZAS`)**: Cada amenaza se conecta a un archivo en `geovisor/capas/`. Añadir una nueva capa es agregar un bloque al diccionario.
4. **Sistema de referencia oficial**: CRTM05 (EPSG:5367) con resolución 10 m o 30 m configurable.
5. **Auditoria de disponibilidad**: La Celda 0 verifica en tiempo real qué capas están disponibles y cuáles están pendientes de publicación.
6. **Contingencia morfológica documentada**: Las capas pendientes se modelan analíticamente desde la geomorfología de La Carpio (cañones Virilla/Torres) para no bloquear el flujo de trabajo.


## 0. Auditoría de Capas Disponibles en el Repositorio GitHub

> ⚠️ **Ejecutar esta celda primero.** Verifica en tiempo real cuáles archivos están publicados en
> `geovisor/capas/` del repositorio canónico (`copernicuscentroamerica/urbano-cr-la-carpio`).
> Las capas pendientes de publicación por el equipo activarán el modo de contingencia morfológica
> documentada en las celdas siguientes.


In [ ]:
#@title 0. Auditoría de disponibilidad de capas en GitHub (ejecutar primero)
import urllib.request
import urllib.error

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/copernicuscentroamerica/urbano-cr-la-carpio/main/geovisor/capas"

# Mapa completo de archivos esperados por el modelo
ARCHIVOS_ESPERADOS = {
    "perimetro_zona_estudio.geojson": {
        "descripcion": "Perimetro operativo homologado de La Carpio",
        "rol": "ROI / Mascara de estudio",
        "subido_por": "Equipo (copernicuscentroamerica)"
    },
    "la_carpio_referencia_osm.geojson": {
        "descripcion": "Contorno de referencia OSM (~67.6 ha)",
        "rol": "Referencia geometrica",
        "subido_por": "Equipo (copernicuscentroamerica)"
    },
    "inundacion_cne.geojson": {
        "descripcion": "Zonas de susceptibilidad de inundacion (CNE)",
        "rol": "Amenaza — Inundacion (peso 30%)",
        "subido_por": "Pendiente de publicacion"
    },
    "cauces_rios.geojson": {
        "descripcion": "Red hidrica — Rios Virilla y Torres",
        "rol": "Amenaza — Cauces fluviales (peso 25%)",
        "subido_por": "Pendiente de publicacion"
    },
    "deslizamientos_cne.geojson": {
        "descripcion": "Coronas y susceptibilidad a deslizamientos (CNE)",
        "rol": "Amenaza — Deslizamientos (peso 20%)",
        "subido_por": "Pendiente de publicacion"
    },
    "fallas_activas.geojson": {
        "descripcion": "Trazas de fallas geologicas activas",
        "rol": "Amenaza — Fallas geologicas (peso 15%)",
        "subido_por": "Pendiente de publicacion"
    },
    "incidentes_historicos.geojson": {
        "descripcion": "Eventos de emergencia historicos (KDE)",
        "rol": "Amenaza — Incidentes historicos (peso 10%)",
        "subido_por": "Pendiente de publicacion"
    },
    "techos_la_carpio.geojson": {
        "descripcion": "Huellas de edificaciones y techos validados",
        "rol": "Capa de exposicion",
        "subido_por": "Pendiente de publicacion"
    }
}

print("=" * 70)
print("AUDITORIA DE CAPAS EN REPOSITORIO GITHUB DEL PROYECTO")
print(f"Repositorio: copernicuscentroamerica/urbano-cr-la-carpio")
print(f"Rama: main  |  Directorio: geovisor/capas/")
print("=" * 70)

disponibilidad = {}
for archivo, meta in ARCHIVOS_ESPERADOS.items():
    url = f"{GITHUB_RAW_BASE}/{archivo}"
    try:
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req, timeout=8) as resp:
            estado = "✅ DISPONIBLE"
            disponibilidad[archivo] = True
    except urllib.error.HTTPError as e:
        if e.code == 404:
            estado = "⚠️  PENDIENTE (404 — no publicado aun)"
        else:
            estado = f"⚠️  ERROR HTTP {e.code}"
        disponibilidad[archivo] = False
    except Exception as e:
        estado = f"⚠️  SIN CONEXION: {e}"
        disponibilidad[archivo] = False

    print(f"\n  Archivo : {archivo}")
    print(f"  Rol     : {meta['rol']}")
    print(f"  Estado  : {estado}")

n_ok = sum(disponibilidad.values())
n_total = len(disponibilidad)
print(f"\n{'='*70}")
print(f"RESUMEN: {n_ok}/{n_total} capas disponibles en el repositorio.")
if n_ok < n_total:
    pendientes = [k for k, v in disponibilidad.items() if not v]
    print(f"Capas pendientes de publicacion por el equipo:")
    for p in pendientes:
        print(f"  - {p}  ({ARCHIVOS_ESPERADOS[p]['subido_por']})")
    print("\nModo: CONTINGENCIA MORFOLOGICA activo para capas pendientes.")
    print("El motor calculara el score usando geometrias de referencia documentadas.")
else:
    print("Modo: DATOS REALES — todas las capas de amenaza estan disponibles.")
print(f"{'='*70}")


## 1. Configuración del Entorno y Librerías

Instalación de dependencias geoespaciales abiertas y conexión opcional con Google Drive para leer
insumos vectoriales institucionales (CNE, MIVAH, INVU, MSJ).


In [ ]:
#@title 1.1 Instalación de dependencias geoespaciales
import subprocess
import sys

packages = [
    "geopandas",
    "rioxarray",
    "rasterio",
    "rasterstats",
    "overturemaps",
    "folium",
    "matplotlib",
    "mapclassify",
    "scipy",
    "shapely"
]

print("Instalando librerias necesarias en Google Colab...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
print("Todas las librerias fueron instaladas exitosamente.")


In [ ]:
#@title 1.2 Importaciones y conexión opcional con Google Drive
import os
import shutil
import zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely.geometry as sgeom
from shapely.ops import unary_union
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from rasterstats import zonal_stats
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
from folium import plugins

# Detectar si estamos en Google Colab
try:
    from google.colab import drive, files
    IN_COLAB = True
    print("Entorno Google Colab detectado.")
except ImportError:
    IN_COLAB = False
    print("Entorno local / Jupyter estandar detectado.")

MOUNT_DRIVE = False #@param {type:"boolean"}
if IN_COLAB and MOUNT_DRIVE:
    drive.mount('/content/drive')
    print("Google Drive conectado correctamente.")


## 2. Definición del Área de Estudio (ROI) y Parámetros

Carga del **perímetro oficial homologado de La Carpio** directamente desde el repositorio GitHub
(`geovisor/capas/perimetro_zona_estudio.geojson`) y configuración de la rejilla de análisis en
**CRTM05 (EPSG:5367)**.

> **📌 Capa confirmada como disponible en GitHub:** `perimetro_zona_estudio.geojson`
> Publicada por el equipo (copernicuscentroamerica/urbano-cr-la-carpio, v2.0.x).
> Bounds aproximados: `[-84.1563, 9.9580, -84.1446, 9.9672]` (~67 ha, 71 vértices).


In [ ]:
#@title 2.1 Carga del perimetro homologado y parametros de la rejilla
RESOLUCION_M = 10 #@param [10, 30] {type:"raw"}
CRS_PROYECTADO = "EPSG:5367"  # CRTM05 oficial de Costa Rica
CRS_GEOGRAFICO = "EPSG:4326"  # WGS84

# URL canonica del perimetro — confirmado disponible en el repositorio GitHub
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/copernicuscentroamerica/urbano-cr-la-carpio/main/geovisor/capas"
URL_PERIMETRO_GITHUB = f"{GITHUB_RAW_BASE}/perimetro_zona_estudio.geojson"

# Bbox de respaldo extraido del perimetro real (evita cero dependencias de red para el ROI)
BBOX_FALLBACK = [-84.156261, 9.958023, -84.144616, 9.967243]

gdf_roi_wgs84 = None

# 1. Intentar descargar el perimetro desde GitHub (disponible)
try:
    print(f"Descargando perimetro oficial desde GitHub...")
    gdf_roi_wgs84 = gpd.read_file(URL_PERIMETRO_GITHUB)
    print(f"  -> Perimetro cargado exitosamente: {len(gdf_roi_wgs84)} poligono(s).")
except Exception as e:
    print(f"  -> Aviso de red: {e}. Usando bounding box de respaldo documentado.")
    gdf_roi_wgs84 = gpd.GeoDataFrame(
        [{"id": "la-carpio", "geometry": sgeom.box(*BBOX_FALLBACK)}], crs=CRS_GEOGRAFICO
    )

# 2. Asegurar CRS WGS84 y reproyectar a CRTM05
if gdf_roi_wgs84.crs is None:
    gdf_roi_wgs84.set_crs(CRS_GEOGRAFICO, inplace=True)
elif str(gdf_roi_wgs84.crs) != CRS_GEOGRAFICO:
    gdf_roi_wgs84 = gdf_roi_wgs84.to_crs(CRS_GEOGRAFICO)

gdf_roi_crtm05 = gdf_roi_wgs84.to_crs(CRS_PROYECTADO)
b_wgs84 = gdf_roi_wgs84.total_bounds
BBOX_LA_CARPIO_WGS84 = [float(b_wgs84[0]), float(b_wgs84[1]), float(b_wgs84[2]), float(b_wgs84[3])]

# 3. Rejilla raster en CRTM05
minx, miny, maxx, maxy = gdf_roi_crtm05.total_bounds
ancho_m = maxx - minx
alto_m  = maxy - miny
cols    = int(np.ceil(ancho_m / RESOLUCION_M))
filas   = int(np.ceil(alto_m  / RESOLUCION_M))
transform_crtm05 = from_bounds(minx, miny, maxx, maxy, cols, filas)

area_ha = gdf_roi_crtm05.geometry.area.sum() / 10_000.0
print(f"\n--- Parametros Operativos ---")
print(f"Perimetro: La Carpio ({area_ha:.1f} ha) | CRTM05 (EPSG:5367)")
print(f"Bbox WGS84: {[round(x,6) for x in BBOX_LA_CARPIO_WGS84]}")
print(f"Rejilla: {filas} filas x {cols} columnas a {RESOLUCION_M} m/px ({filas*cols:,} celdas totales)")


## 3. Catálogo Extensible de Amenazas (`CATALOGO_AMENAZAS`) Basado en GitHub

### Estado actual de las capas en el repositorio (auditado 2026-11-11)

| Archivo en `geovisor/capas/` | Rol en el Catálogo | Estado |
|---|---|:---:|
| `perimetro_zona_estudio.geojson` | ROI / Área de estudio | ✅ Disponible |
| `la_carpio_referencia_osm.geojson` | Referencia geométrica | ✅ Disponible |
| `inundacion_cne.geojson` | Amenaza — Inundación (30%) | ⚠️ Pendiente |
| `cauces_rios.geojson` | Amenaza — Cauces fluviales (25%) | ⚠️ Pendiente |
| `deslizamientos_cne.geojson` | Amenaza — Deslizamientos (20%) | ⚠️ Pendiente |
| `fallas_activas.geojson` | Amenaza — Fallas geológicas (15%) | ⚠️ Pendiente |
| `incidentes_historicos.geojson` | Amenaza — Incidentes históricos (10%) | ⚠️ Pendiente |
| `techos_la_carpio.geojson` | Exposición — Edificaciones | ⚠️ Pendiente |

> **⚠️ Nota:** Las 5 capas de amenaza y la capa de techos están en preparación por el equipo
> interinstitucional. Mientras tanto, el motor activa la **contingencia morfológica documentada**:
> geometrías derivadas de la geomorfología conocida de La Carpio (cañones Río Virilla / Torres).
> Cuando el equipo publique los archivos reales, el catálogo los detectará y usará automáticamente.

**Lógica de carga por capa (en orden de prioridad):**
1. Archivo local en el repositorio clonado (`geovisor/capas/...`)
2. Descarga directa desde GitHub Raw URL
3. Contingencia morfológica documentada (modo de referencia)


In [ ]:
#@title 3.1 Diccionario modular de amenazas vinculado al repositorio GitHub
# El GITHUB_RAW_BASE ya fue definido en la celda anterior
CATALOGO_AMENAZAS = {
    "inundacion_cne": {
        "descripcion": "Zonas de susceptibilidad de inundacion (CNE / MIVAH)",
        "tipo": "vector_poligono",
        "ruta_local": "geovisor/capas/inundacion_cne.geojson",
        "url_remota": f"{GITHUB_RAW_BASE}/inundacion_cne.geojson",
        "peso": 0.30,
        "valor_severidad": 1.0,
        "estado_github": "en_preparacion",
        "institucion": "CNE / MIVAH"
    },
    "cauces_rios": {
        "descripcion": "Red hidrica — Rios Virilla y Torres (zona proteccion fluvial)",
        "tipo": "vector_linea_buffer",
        "ruta_local": "geovisor/capas/cauces_rios.geojson",
        "url_remota": f"{GITHUB_RAW_BASE}/cauces_rios.geojson",
        "buffer_m": [30, 60, 100],
        "pesos_buffer": [1.0, 0.6, 0.3],
        "peso": 0.25,
        "estado_github": "en_preparacion",
        "institucion": "CNE / MIVAH / SINAC"
    },
    "coronas_deslizamiento": {
        "descripcion": "Taludes de canones fluviales y susceptibilidad a deslizamientos",
        "tipo": "vector_poligono",
        "ruta_local": "geovisor/capas/deslizamientos_cne.geojson",
        "url_remota": f"{GITHUB_RAW_BASE}/deslizamientos_cne.geojson",
        "peso": 0.20,
        "valor_severidad": 1.0,
        "estado_github": "en_preparacion",
        "institucion": "CNE"
    },
    "fallas_geologicas": {
        "descripcion": "Trazas de fallas geologicas activas y zonas de influencia",
        "tipo": "vector_linea_buffer",
        "ruta_local": "geovisor/capas/fallas_activas.geojson",
        "url_remota": f"{GITHUB_RAW_BASE}/fallas_activas.geojson",
        "buffer_m": [50, 150],
        "pesos_buffer": [1.0, 0.5],
        "peso": 0.15,
        "estado_github": "en_preparacion",
        "institucion": "CNE / OVSICORI"
    },
    "incidentes_historicos": {
        "descripcion": "Eventos de emergencia historicos registrados (densidad kernel)",
        "tipo": "vector_puntos_densidad",
        "ruta_local": "geovisor/capas/incidentes_historicos.geojson",
        "url_remota": f"{GITHUB_RAW_BASE}/incidentes_historicos.geojson",
        "radio_kernel_m": 150,
        "peso": 0.10,
        "estado_github": "en_preparacion",
        "institucion": "CNE / 9-1-1 / MSJ"
    }
}

print(f"Catalogo de amenazas configurado ({len(CATALOGO_AMENAZAS)} capas):")
print(f"{'Clave':<25} {'Peso':>6}  {'Estado en GitHub':<20}  {'Descripcion'}")
print("-" * 90)
for k, v in CATALOGO_AMENAZAS.items():
    icono = "✅" if v["estado_github"] == "disponible" else "⚠️"
    print(f"  {k:<23} {v['peso']*100:>5.0f}%  {icono} {v['estado_github']:<18}  {v['descripcion']}")


In [ ]:
#@title 3.2 Cargador de capas desde GitHub (con modo contingencia morfologica)
import urllib.request

def obtener_capa_amenaza(nombre_clave, config, gdf_roi_crtm05):
    """
    Carga una capa de amenaza en orden de prioridad:
    1. Archivo local del repositorio clonado
    2. URL remota del repositorio GitHub publico
    3. Contingencia: geometria morfologica documentada de La Carpio
    """
    ruta_local = config.get("ruta_local")
    url_remota  = config.get("url_remota")

    # Prioridad 1: Archivo local (repositorio clonado en la maquina o Drive)
    if ruta_local and os.path.exists(ruta_local):
        print(f"  [LOCAL] '{nombre_clave}' cargado desde: {ruta_local}")
        gdf = gpd.read_file(ruta_local)
        if str(gdf.crs) != CRS_PROYECTADO:
            gdf = gdf.to_crs(CRS_PROYECTADO)
        return gdf.clip(gdf_roi_crtm05), "local"

    # Prioridad 2: Descarga directa desde GitHub Raw
    if url_remota:
        try:
            print(f"  [GITHUB] Descargando '{nombre_clave}' desde GitHub...")
            gdf = gpd.read_file(url_remota)
            if str(gdf.crs) != CRS_PROYECTADO:
                gdf = gdf.to_crs(CRS_PROYECTADO)
            print(f"  [GITHUB] '{nombre_clave}' disponible: {len(gdf)} geometrias.")
            return gdf.clip(gdf_roi_crtm05), "github"
        except Exception as e:
            print(f"  [INFO] '{nombre_clave}' no publicado aun en GitHub ({e})")

    # Prioridad 3: Contingencia morfologica documentada
    print(f"  [CONTINGENCIA] Modelando '{nombre_clave}' con geometria de referencia de La Carpio.")
    b = gdf_roi_crtm05.total_bounds
    minx, miny, maxx, maxy = b
    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2

    if config["tipo"] == "vector_linea_buffer" and "cauce" in nombre_clave:
        # Rios Virilla (norte) y Torres (sur) encajonan a La Carpio
        l_norte = sgeom.LineString([(minx - 100, maxy - 120), (cx, maxy - 80), (maxx + 100, maxy - 140)])
        l_sur   = sgeom.LineString([(minx - 100, miny + 100), (cx, miny + 130), (maxx + 100, miny + 90)])
        return gpd.GeoDataFrame({"nombre": ["Rio Virilla", "Rio Torres"]}, geometry=[l_norte, l_sur], crs=CRS_PROYECTADO), "contingencia"

    elif config["tipo"] == "vector_linea_buffer" and "falla" in nombre_clave:
        l_falla = sgeom.LineString([(minx + 200, miny - 50), (cx + 100, cy), (maxx - 150, maxy + 50)])
        return gpd.GeoDataFrame({"nombre": ["Falla inferida"]}, geometry=[l_falla], crs=CRS_PROYECTADO), "contingencia"

    elif "inundacion" in nombre_clave:
        # Franjas de inundacion en los bordes norte y sur del canon
        poly_n = sgeom.box(minx, maxy - 250, maxx, maxy)
        poly_s = sgeom.box(minx, miny,       maxx, miny + 200)
        return gpd.GeoDataFrame({"severidad": [1.0, 1.0]}, geometry=[poly_n, poly_s], crs=CRS_PROYECTADO), "contingencia"

    elif "deslizamiento" in nombre_clave or "corona" in nombre_clave:
        # Taludes superiores del canon
        talud1 = sgeom.Polygon([(minx + 100, maxy - 300), (maxx - 200, maxy - 280), (maxx - 150, maxy - 150), (minx + 50, maxy - 180)])
        talud2 = sgeom.Polygon([(minx + 150, miny + 120), (maxx - 100, miny + 150), (maxx - 50, miny + 280), (minx + 120, miny + 250)])
        return gpd.GeoDataFrame({"severidad": [1.0, 1.0]}, geometry=[talud1, talud2], crs=CRS_PROYECTADO), "contingencia"

    elif config["tipo"] == "vector_puntos_densidad":
        np.random.seed(42)
        n = 35
        xs = np.random.uniform(minx + 200, maxx - 200, n)
        ys = np.random.uniform(miny + 150, maxy - 150, n)
        pts = [sgeom.Point(x, y) for x, y in zip(xs, ys)]
        return gpd.GeoDataFrame({"evento": [f"Inc_{i+1}" for i in range(n)]}, geometry=pts, crs=CRS_PROYECTADO), "contingencia"

    return gpd.GeoDataFrame(geometry=[], crs=CRS_PROYECTADO), "contingencia"

print("Cargador de capas de amenaza configurado con 3 niveles de prioridad.")


## 4. Motor de Normalización y Álgebra de Mapas

Conversión de todas las amenazas hacia la rejilla ráster común (CRTM05 / resolución configurable),
normalización en rango $[0, 1]$ y cálculo del **Score Multiamenaza Ponderado**.

La fórmula aplicada es:

$$\text{Score} = \sum_{i=1}^{N} \frac{w_i}{\sum w_j} \cdot \hat{A}_i$$

donde $\hat{A}_i$ es la amenaza $i$ normalizada al rango $[0, 1]$ y $w_i$ su peso configurado en el catálogo.


In [ ]:
#@title 4.1 Rasterizacion y calculo del Score Multiamenaza Ponderado
rasters_amenazas = {}
pesos_normalizados = {}
modos_carga = {}

suma_pesos = sum(cfg["peso"] for cfg in CATALOGO_AMENAZAS.values())

print("Procesando capas del catalogo de amenazas...")
print("-" * 60)

for nombre, cfg in CATALOGO_AMENAZAS.items():
    peso_norm = cfg["peso"] / suma_pesos
    pesos_normalizados[nombre] = peso_norm
    tipo = cfg["tipo"]

    gdf_insumo, modo = obtener_capa_amenaza(nombre, cfg, gdf_roi_crtm05)
    modos_carga[nombre] = modo

    arr_capa = np.zeros((filas, cols), dtype=np.float32)

    if tipo == "vector_poligono":
        if not gdf_insumo.empty:
            shapes = [(geom, float(cfg.get("valor_severidad", 1.0)))
                      for geom in gdf_insumo.geometry if geom is not None and geom.is_valid]
            if shapes:
                arr_capa = rasterize(shapes=shapes, out_shape=(filas, cols),
                                     transform=transform_crtm05, fill=0, dtype=np.float32)

    elif tipo == "vector_linea_buffer":
        if not gdf_insumo.empty:
            buffers    = cfg.get("buffer_m", [50])
            pesos_buf  = cfg.get("pesos_buffer", [1.0])
            for dist, p_buf in sorted(zip(buffers, pesos_buf), key=lambda x: x[0], reverse=True):
                buf_geoms = gdf_insumo.geometry.buffer(dist)
                shapes    = [(g, float(p_buf)) for g in buf_geoms if g is not None and g.is_valid]
                if shapes:
                    arr_tmp   = rasterize(shapes=shapes, out_shape=(filas, cols),
                                          transform=transform_crtm05, fill=0, dtype=np.float32)
                    arr_capa  = np.maximum(arr_capa, arr_tmp)

    elif tipo == "vector_puntos_densidad":
        if not gdf_insumo.empty:
            shapes    = [(geom, 1.0) for geom in gdf_insumo.geometry]
            pts_rast  = rasterize(shapes=shapes, out_shape=(filas, cols),
                                  transform=transform_crtm05, fill=0, dtype=np.float32)
            sigma_px  = cfg.get("radio_kernel_m", 150) / RESOLUCION_M
            kde       = gaussian_filter(pts_rast, sigma=sigma_px)
            if kde.max() > 0:
                arr_capa = kde / kde.max()

    if arr_capa.max() > 0:
        arr_capa = (arr_capa - arr_capa.min()) / (arr_capa.max() - arr_capa.min())

    rasters_amenazas[nombre] = arr_capa
    icono = "🌐" if modo == "github" else ("📁" if modo == "local" else "🔶")
    print(f"  {icono} [{nombre}] Peso={peso_norm:.2f} | "
          f"Celdas activas={int((arr_capa > 0).sum()):,} | Fuente={modo}")

# Score ponderado total
score_multiamenaza = np.zeros((filas, cols), dtype=np.float32)
for nombre, arr in rasters_amenazas.items():
    score_multiamenaza += pesos_normalizados[nombre] * arr

print(f"\nScore Multiamenaza calculado exitosamente.")
print(f"Min={score_multiamenaza.min():.3f} | Max={score_multiamenaza.max():.3f} | "
      f"Media={score_multiamenaza.mean():.3f}")

# Reporte de fuentes de datos
n_real = sum(1 for m in modos_carga.values() if m in ("github","local"))
n_cont = sum(1 for m in modos_carga.values() if m == "contingencia")
print(f"\nFuentes usadas: {n_real} capa(s) REAL(ES) del repositorio + {n_cont} en CONTINGENCIA MORFOLOGICA.")
if n_cont > 0:
    print("  -> Las capas en contingencia se reemplazaran automaticamente cuando el equipo las publique en geovisor/capas/.")


In [ ]:
#@title 4.2 Reclasificacion en 5 niveles estandar (compatible con GEE original)
# Esquema de colores y limites del script GEE original: 5 intervalos iguales
s_min, s_max = float(score_multiamenaza.min()), float(score_multiamenaza.max())
delta = (s_max - s_min) / 5.0
limites_clases = [s_min + i * delta for i in range(1, 5)]

score_clasificado = np.ones((filas, cols), dtype=np.uint8)
score_clasificado[score_multiamenaza >= limites_clases[0]] = 2
score_clasificado[score_multiamenaza >= limites_clases[1]] = 3
score_clasificado[score_multiamenaza >= limites_clases[2]] = 4
score_clasificado[score_multiamenaza >= limites_clases[3]] = 5

CLASES_INFO = {
    1: {"nombre": "Muy Bajo", "color": "#FFFFE0"},
    2: {"nombre": "Bajo",     "color": "#FFFF00"},
    3: {"nombre": "Medio",    "color": "#FFA500"},
    4: {"nombre": "Alto",     "color": "#FF0000"},
    5: {"nombre": "Muy Alto", "color": "#8B4513"},
}

print("Umbrales de clasificacion (intervalos iguales):")
for idx, lim in enumerate(limites_clases, start=1):
    print(f"  Clase {idx} ({CLASES_INFO[idx]['nombre']:8s}) <= {lim:.3f} < Clase {idx+1} ({CLASES_INFO[idx+1]['nombre']})")


## 5. Carga de Edificaciones y Reducción Zonal Vectorial

Descarga de huellas de edificaciones desde el repositorio GitHub (`geovisor/capas/techos_la_carpio.geojson`)
o descarga abierta vía **Overture Maps** (Google Open Buildings + OSM). Se calcula la **amenaza promedio
por huella de edificio**.

> **Estado actual:** `techos_la_carpio.geojson` está en preparación. El cuaderno usará Overture Maps
> como primera alternativa abierta.


In [ ]:
#@title 5.1 Obtencion de huellas de edificaciones
FUENTE_EDIFICIOS = "Repositorio GitHub (geovisor/capas/)" #@param ["Repositorio GitHub (geovisor/capas/)", "Overture Maps (Descarga abierta)", "Archivo local / Google Drive"]
RUTA_EDIFICIOS_LOCAL  = "geovisor/capas/techos_la_carpio.geojson" #@param {type:"string"}
URL_EDIFICIOS_GITHUB  = f"{GITHUB_RAW_BASE}/techos_la_carpio.geojson"

gdf_edificios = None
fuente_edificios_usada = None

# Prioridad 1: Repositorio GitHub
if FUENTE_EDIFICIOS == "Repositorio GitHub (geovisor/capas/)":
    if os.path.exists(RUTA_EDIFICIOS_LOCAL):
        print(f"[LOCAL] Cargando techos desde archivo local: {RUTA_EDIFICIOS_LOCAL}")
        gdf_edificios = gpd.read_file(RUTA_EDIFICIOS_LOCAL)
        fuente_edificios_usada = "local"
    else:
        try:
            print(f"[GITHUB] Descargando techos desde: {URL_EDIFICIOS_GITHUB}")
            gdf_edificios = gpd.read_file(URL_EDIFICIOS_GITHUB)
            fuente_edificios_usada = "github"
            print(f"  -> {len(gdf_edificios)} huellas descargadas desde GitHub.")
        except Exception:
            print("[INFO] 'techos_la_carpio.geojson' aun no publicado. Intentando Overture Maps...")

# Prioridad 2: Overture Maps (Google Open Buildings + OSM)
if gdf_edificios is None or gdf_edificios.empty:
    try:
        import overturemaps
        print("[OVERTURE] Consultando Google Open Buildings / OSM para La Carpio...")
        table = overturemaps.record_batch_reader("building", BBOX_LA_CARPIO_WGS84).read_all()
        gdf_ov = gpd.GeoDataFrame.from_arrow(table)
        if not gdf_ov.empty and "geometry" in gdf_ov.columns:
            gdf_ov.set_crs(CRS_GEOGRAFICO, inplace=True)
            gdf_edificios = gdf_ov.to_crs(CRS_PROYECTADO).clip(gdf_roi_crtm05)
            fuente_edificios_usada = "overture"
            print(f"  -> {len(gdf_edificios)} edificios descargados desde Overture Maps.")
    except Exception as e:
        print(f"[INFO] Overture no disponible: {e}. Creando capa de referencia.")

# Asegurar CRS
if gdf_edificios is not None and not gdf_edificios.empty:
    if str(gdf_edificios.crs) != CRS_PROYECTADO:
        gdf_edificios = gdf_edificios.to_crs(CRS_PROYECTADO)

# Prioridad 3: Fallback demostrativo (120 edificios representativos)
if gdf_edificios is None or gdf_edificios.empty:
    print("[CONTINGENCIA] Generando capa de edificios representativos para La Carpio (n=120).")
    np.random.seed(101)
    n_edif = 120
    b = gdf_roi_crtm05.total_bounds
    xs  = np.random.uniform(b[0] + 300, b[2] - 300, n_edif)
    ys  = np.random.uniform(b[1] + 250, b[3] - 250, n_edif)
    polys = [sgeom.box(x - w/2, y - h/2, x + w/2, y + h/2)
             for x, y, w, h in zip(xs, ys,
                                    np.random.uniform(8, 20, n_edif),
                                    np.random.uniform(8, 18, n_edif))]
    gdf_edificios = gpd.GeoDataFrame({"id_edificio": range(1, n_edif + 1)}, geometry=polys, crs=CRS_PROYECTADO)
    fuente_edificios_usada = "contingencia"

print(f"\nEdificaciones listas: {len(gdf_edificios)} huellas | Fuente: {fuente_edificios_usada}")


In [ ]:
#@title 5.2 Reduccion zonal de amenaza por poligono de edificio (rasterstats)
print("Ejecutando estadistica zonal vectorial...")

stats = zonal_stats(
    vectors=gdf_edificios,
    raster=score_multiamenaza,
    affine=transform_crtm05,
    stats=["mean", "max"],
    nodata=-9999
)

gdf_edificios["score_medio"] = [s["mean"] if s["mean"] is not None else 0.0 for s in stats]
gdf_edificios["score_max"]   = [s["max"]  if s["max"]  is not None else 0.0 for s in stats]

def clasificar_score(val):
    if   val >= limites_clases[3]: return (5, "Muy Alto", "#8B4513")
    elif val >= limites_clases[2]: return (4, "Alto",     "#FF0000")
    elif val >= limites_clases[1]: return (3, "Medio",    "#FFA500")
    elif val >= limites_clases[0]: return (2, "Bajo",     "#FFFF00")
    else:                          return (1, "Muy Bajo", "#FFFFE0")

res_clases = gdf_edificios["score_medio"].apply(clasificar_score)
gdf_edificios["clase_id"]     = [r[0] for r in res_clases]
gdf_edificios["clase_nombre"] = [r[1] for r in res_clases]
gdf_edificios["color_hex"]    = [r[2] for r in res_clases]

print(f"Estadistica zonal completada: {len(gdf_edificios)} edificaciones clasificadas.")


## 6. Reporte de Distribución e Impacto Territorial

Resumen cuantitativo del número de edificaciones en cada rango de amenaza y gráfico de barras.


In [ ]:
#@title 6.1 Tabla y grafico de distribucion por nivel de amenaza
conteo_clases = gdf_edificios["clase_nombre"].value_counts().reindex(
    ["Muy Bajo", "Bajo", "Medio", "Alto", "Muy Alto"], fill_value=0
)
pcts = (conteo_clases / len(gdf_edificios)) * 100

df_resumen = pd.DataFrame({
    "Nivel de Amenaza": conteo_clases.index,
    "Edificaciones": conteo_clases.values,
    "Porcentaje (%)": pcts.values.round(1)
})

print("EXPOSICION DE EDIFICACIONES EN LA CARPIO — DISTRIBUCION POR NIVEL DE AMENAZA")
print("=" * 60)
print(df_resumen.to_string(index=False))
print(f"\nTotal edificaciones analizadas: {len(gdf_edificios)}")
print(f"Fuente edificaciones: {fuente_edificios_usada}")
print(f"Fuentes amenaza: {', '.join(set(modos_carga.values()))}")

colores_grafico = [CLASES_INFO[i]["color"] for i in range(1, 6)]
fig, ax = plt.subplots(figsize=(8, 4.5))
barras  = ax.bar(df_resumen["Nivel de Amenaza"], df_resumen["Edificaciones"],
                 color=colores_grafico, edgecolor="black", linewidth=0.8)
ax.set_title("Edificaciones por Nivel de Multiamenaza Ponderada\n"
             "(Asentamiento La Carpio, San José, Costa Rica)", fontsize=12, fontweight="bold")
ax.set_xlabel("Nivel de Amenaza", fontsize=10)
ax.set_ylabel("Cantidad de Edificaciones", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.6)
for b in barras:
    yv = b.get_height()
    ax.text(b.get_x() + b.get_width()/2.0, yv + 1,
            f"{int(yv)}\n({yv/len(gdf_edificios)*100:.1f}%)",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("distribucion_multiamenaza_edificaciones.png", dpi=200)
plt.show()


## 7. Visualizador Interactivo en Mapa (Folium)

Mapa interactivo con capas activables: polígonos de edificios coloreados por nivel de amenaza,
perímetro del área de estudio y leyenda flotante oficial.


In [ ]:
#@title 7.1 Construccion del mapa web interactivo (Folium)
gdf_edif_wgs84 = gdf_edificios.to_crs(CRS_GEOGRAFICO)
gdf_roi_wgs84_disp = gdf_roi_wgs84.copy()

centro_lat = (BBOX_LA_CARPIO_WGS84[1] + BBOX_LA_CARPIO_WGS84[3]) / 2
centro_lon = (BBOX_LA_CARPIO_WGS84[0] + BBOX_LA_CARPIO_WGS84[2]) / 2

m = folium.Map(location=[centro_lat, centro_lon], zoom_start=15, tiles="CartoDB positron")

# Perimetro del area de estudio
fg_perimetro = folium.FeatureGroup(name="Perimetro de estudio (GitHub)")
folium.GeoJson(
    gdf_roi_wgs84_disp.__geo_interface__,
    style_function=lambda x: {
        "fillColor": "#007F8B", "color": "#14283F",
        "weight": 2.5, "fillOpacity": 0.08
    },
    tooltip="Perimetro La Carpio — homologado (GitHub)"
).add_to(fg_perimetro)
fg_perimetro.add_to(m)

# Edificaciones clasificadas
fg_edif = folium.FeatureGroup(name="Edificaciones — Score Multiamenaza")
for _, row in gdf_edif_wgs84.iterrows():
    popup_txt = (
        f"<b>ID:</b> {row.get('id_edificio', row.name)}<br>"
        f"<b>Score:</b> {row['score_medio']:.3f}<br>"
        f"<b>Nivel:</b> <span style='color:{row['color_hex']};font-weight:bold;'>{row['clase_nombre']}</span>"
    )
    folium.GeoJson(
        row.geometry,
        style_function=lambda x, color=row['color_hex']: {
            "fillColor": color, "color": "#333333", "weight": 1, "fillOpacity": 0.82
        },
        tooltip=f"{row['clase_nombre']} (Score: {row['score_medio']:.2f})",
        popup=folium.Popup(popup_txt, max_width=240)
    ).add_to(fg_edif)
fg_edif.add_to(m)

# Leyenda flotante
leyenda_html = (
    '<div style="position:fixed;bottom:30px;left:30px;width:195px;'
    'background-color:white;border:2px solid #555;z-index:9999;'
    'font-size:12px;padding:10px;border-radius:5px;box-shadow:2px 2px 6px rgba(0,0,0,0.3);">'
    '<b style="font-size:13px;">Multiamenaza La Carpio</b><br>'
    '<i style="background:#FFFFE0;width:14px;height:14px;float:left;margin-right:7px;border:1px solid #aaa;"></i>Muy Bajo<br>'
    '<i style="background:#FFFF00;width:14px;height:14px;float:left;margin-right:7px;border:1px solid #aaa;"></i>Bajo<br>'
    '<i style="background:#FFA500;width:14px;height:14px;float:left;margin-right:7px;border:1px solid #aaa;"></i>Medio<br>'
    '<i style="background:#FF0000;width:14px;height:14px;float:left;margin-right:7px;border:1px solid #aaa;"></i>Alto<br>'
    '<i style="background:#8B4513;width:14px;height:14px;float:left;margin-right:7px;border:1px solid #aaa;"></i>Muy Alto<br>'
    '</div>'
)
m.get_root().html.add_child(folium.Element(leyenda_html))
folium.LayerControl(collapsed=False).add_to(m)

m.save("mapa_multiamenaza_la_carpio.html")
print("Mapa interactivo guardado: 'mapa_multiamenaza_la_carpio.html'")
m


## 8. Exportación y Empaquetado Automático (.ZIP)

Generación de archivos estándar para SIG de escritorio (QGIS, ArcGIS):
- **GeoTIFF** continuo y clasificado en CRTM05 (5367).
- **GeoPackage / GeoJSON** de edificaciones con nivel de amenaza y color HEX.
- **Archivo `.zip`** con todos los resultados para descarga directa desde Colab.


In [ ]:
#@title 8.1 Exportacion de resultados y creacion del paquete ZIP
os.makedirs("salidas_la_carpio", exist_ok=True)

meta_base = {
    "driver": "GTiff", "height": filas, "width": cols, "count": 1,
    "crs": CRS_PROYECTADO, "transform": transform_crtm05, "compress": "lzw"
}

# GeoTIFF continuo
meta_cont = {**meta_base, "dtype": np.float32}
with rasterio.open("salidas_la_carpio/score_multiamenaza_continuo.tif", "w", **meta_cont) as dst:
    dst.write(score_multiamenaza, 1)

# GeoTIFF clasificado
meta_clas = {**meta_base, "dtype": np.uint8}
with rasterio.open("salidas_la_carpio/score_multiamenaza_clasificado.tif", "w", **meta_clas) as dst:
    dst.write(score_clasificado, 1)

# Vectoriales
gdf_edificios.to_file("salidas_la_carpio/edificaciones_amenaza.gpkg", layer="edificaciones", driver="GPKG")
gdf_edificios.to_crs(CRS_GEOGRAFICO).to_file("salidas_la_carpio/edificaciones_amenaza.geojson", driver="GeoJSON")

# Copiar graficos y mapa
for src_file in ["distribucion_multiamenaza_edificaciones.png", "mapa_multiamenaza_la_carpio.html"]:
    if os.path.exists(src_file):
        shutil.copy(src_file, "salidas_la_carpio/")

# ZIP final
zip_salida = "resultados_multiamenaza_la_carpio.zip"
with zipfile.ZipFile(zip_salida, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files_list in os.walk("salidas_la_carpio"):
        for f in files_list:
            p = os.path.join(root, f)
            zipf.write(p, arcname=os.path.relpath(p, "salidas_la_carpio"))

print(f"Paquete ZIP generado: {zip_salida}")
print("Contenido:")
for item in sorted(os.listdir("salidas_la_carpio")):
    print(f"  - {item}")

if IN_COLAB:
    files.download(zip_salida)
